In [2]:
import os
import json
import numpy as np
import pandas as pd
import sympy as sp

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
MODELSDIR = CONFIGS['filepaths']['models']
SRMODELS  = CONFIGS['experiments']['sr']['optimizedeqs']

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)

regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in regdf.iterrows()}

ORDER  = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}

MU    = STATS['tp_mean']
SIGMA = STATS['tp_std']
ZMIN  = -MU / SIGMA

VARMETA = {
    'bl':    {'sym':r'B_L','unit':'K','mean':STATS['bl_mean'],'std':STATS['bl_std']},
    'rh':    {'sym':r'\widehat{\mathrm{RH}}','unit':'%','mean':STATS['rh_mean'],'std':STATS['rh_std']},
    'thetae':{'sym':r'\widehat{\theta_e}','unit':'K','mean':STATS['thetae_mean'],'std':STATS['thetae_std']},
    'thetaestar':{'sym':r'\widehat{\theta_e^*}','unit':'K','mean':STATS['thetaestar_mean'],'std':STATS['thetaestar_std']},
    'shf':   {'sym':r'\mathrm{SHF}','unit':r'W m$^{-2}$','mean':STATS['shf_mean'],'std':STATS['shf_std']},
    'lhf':   {'sym':r'\mathrm{LHF}','unit':r'W m$^{-2}$','mean':STATS['lhf_mean'],'std':STATS['lhf_std']}}

print(f'Precipitation (log1p-transformed):  mu = {MU:.6f},  sigma = {SIGMA:.6f},  z_0 = {ZMIN:.4f}')
print()
for var,meta in VARMETA.items():
    print(f'{var:>12}: mu = {meta["mean"]:>12.6f},  sigma = {meta["std"]:>10.6f}  [{meta["unit"]}]')

## General prediction framework

Each SR equation learns a latent activation $f(\hat{\mathbf{x}})$ evaluated on standardized predictors $\hat{\mathbf{x}}$. Physical precipitation $P$ (mm) is recovered through two transformations:

$$
z = z_0 + \max\!\big(f(\hat{\mathbf{x}}),\; 0\big), \qquad P = \max\!\Big(\exp\!\big(\mu + \sigma\, z\big) - 1,\; 0\Big),
$$

where $\mu$ and $\sigma$ are the training-set mean and standard deviation of $\log(1 + P)$, and $z_0 = -\mu / \sigma$ is the standardized value corresponding to zero precipitation.

### Key simplification

When $f(\hat{\mathbf{x}}) > 0$, substituting $z = z_0 + f(\hat{\mathbf{x}})$:

$$
P = \exp\!\Big(\mu + \sigma\big(z_0 + f(\hat{\mathbf{x}})\big)\Big) - 1
  = \exp\!\bigg(\mu + \sigma\bigg(\!\!-\frac{\mu}{\sigma} + f(\hat{\mathbf{x}})\bigg)\bigg) - 1
  = \exp\!\Big(\sigma \cdot f(\hat{\mathbf{x}})\Big) - 1.
$$

When $f(\hat{\mathbf{x}}) \leq 0$, $z = z_0$ and $P = \exp(\mu + \sigma z_0) - 1 = \exp(0) - 1 = 0$.

Therefore, the full prediction reduces to:

$$
\boxed{P = \max\!\Big(\exp\!\big(\sigma \cdot f(\hat{\mathbf{x}})\big) - 1,\; 0\Big).}
$$

The training-set mean $\mu$ drops out entirely — it only sets the zero-crossing point. For converting to physical space, we only need $\sigma$ (the precipitation standard deviation) and the per-variable standardization constants $\mu_X$, $\sigma_X$.

## SR-BL: boundary layer depth → precipitation

### Normalized form

$$
f(\hat{B}_L) = (\hat{B}_L + a)^3 + b, \qquad \hat{B}_L = \frac{B_L - \mu_B}{\sigma_B},
$$

where $\mu_B$ and $\sigma_B$ are the training-set mean and standard deviation of boundary layer depth.

### Step 1: substitute the standardization

$$
f = \left(\frac{B_L - \mu_B}{\sigma_B} + a\right)^3 + b = \left(\frac{B_L - \underbrace{(\mu_B - a\,\sigma_B)}_{B_L^*}}{\sigma_B}\right)^3 + b.
$$

Define the **critical BL depth** in physical units:

$$
B_L^* \;\equiv\; \mu_B - a\,\sigma_B.
$$

This is the value of $B_L$ at which the cubic term is zero.

### Step 2: multiply by $\sigma$ and collect constants

$$
\sigma \cdot f = \frac{\sigma}{\sigma_B^3}\,(B_L - B_L^*)^3 + \sigma\, b.
$$

Define two physical constants:

$$
\alpha \;\equiv\; \frac{\sigma}{\sigma_B^3}, \qquad \beta \;\equiv\; \sigma\, b.
$$

Note that $\sigma$ here is the **precipitation** standard deviation, while $\sigma_B$ is the **BL** standard deviation — they are different quantities with different units.

### Step 3: physical-space equation

$$
\boxed{P(B_L) = \max\!\Big(\exp\!\big(\alpha\,(B_L - B_L^*)^3 + \beta\big) - 1,\; 0\Big).}
$$

Three physical constants with clear interpretations:

| Constant | Meaning | Formula |
|:--------:|:--------|:--------|
| $B_L^*$ | Critical BL depth (zero of the cubic) | $\mu_B - a\,\sigma_B$ |
| $\alpha$ | Exponential sensitivity to BL departures from $B_L^*$ | $\sigma / \sigma_B^3$ |
| $\beta$ | Log-space offset controlling overall magnitude | $\sigma \cdot b$ |

### Precipitation onset

$P > 0$ when $\alpha\,(B_L - B_L^*)^3 + \beta > 0$, i.e.:

$$
B_L > B_L^* - \left(\frac{\beta}{\alpha}\right)^{1/3} = B_L^* - b^{1/3}\,\sigma_B.
$$

In [ ]:
name = 'sr_bl_eq'
C = REGISTRY[name]['constants']
a,b = C['a'],C['b']
mu_B,sig_B = VARMETA['bl']['mean'],VARMETA['bl']['std']

BLstar = mu_B - a*sig_B
alpha  = SIGMA / sig_B**3
beta   = SIGMA * b
onset  = BLstar - b**(1/3) * sig_B

print(f'SR-BL physical constants (a={a}, b={b}):')
print(f'  B_L*  = mu_B - a*sig_B        = {BLstar:.4f} K')
print(f'  alpha = sigma / sig_B^3        = {alpha:.6f} K^-3')
print(f'  beta  = sigma * b              = {beta:.6f}')
print(f'  Onset = B_L* - b^(1/3)*sig_B   = {onset:.4f} K')

## SR-ATM: atmospheric thermodynamics → precipitation

### Normalized form

$$
f(\hat{R}, \hat{\theta}_e, \hat{\theta}_e^*) = a \cdot \Big(\max\!\big(\hat{R},\; \hat{\theta}_e - b\,\hat{\theta}_e^* - c\big)\Big)^3,
$$

where the standardized variables are:

$$
\hat{R} = \frac{R - \mu_R}{\sigma_R}, \qquad \hat{\theta}_e = \frac{\theta_e - \mu_e}{\sigma_e}, \qquad \hat{\theta}_e^* = \frac{\theta_e^* - \mu_{e^*}}{\sigma_{e^*}}.
$$

### Step 1: convert the max argument to physical space

The equation selects the larger of two branches. We convert each separately.

**Branch 1** (humidity-dominated):

$$
\hat{R} = \frac{R - \mu_R}{\sigma_R}.
$$

**Branch 2** (instability-dominated):

$$
\hat{\theta}_e - b\,\hat{\theta}_e^* - c = \frac{\theta_e - \mu_e}{\sigma_e} - b\,\frac{\theta_e^* - \mu_{e^*}}{\sigma_{e^*}} - c.
$$

Collecting terms:

$$
= \frac{1}{\sigma_e}\,\theta_e - \frac{b}{\sigma_{e^*}}\,\theta_e^* - \underbrace{\left(\frac{\mu_e}{\sigma_e} - \frac{b\,\mu_{e^*}}{\sigma_{e^*}} + c\right)}_{\gamma_0}.
$$

### Step 2: define physical-space constants

To make the two branches commensurate inside the max, factor out a common scale. Branch 1 naturally has scale $1/\sigma_R$, while Branch 2 has scale $1/\sigma_e$ on $\theta_e$. Since these differ, we keep the equation in a **hybrid form** — the max operates on standardized-scale quantities, and we absorb everything into the outer cube and coefficient.

Define:

$$
\gamma_1 \equiv \frac{1}{\sigma_R}, \qquad \gamma_2 \equiv \frac{1}{\sigma_e}, \qquad \gamma_3 \equiv \frac{b}{\sigma_{e^*}},
$$

$$
\gamma_4 \equiv \frac{\mu_R}{\sigma_R}, \qquad \gamma_0 \equiv \frac{\mu_e}{\sigma_e} - \frac{b\,\mu_{e^*}}{\sigma_{e^*}} + c.
$$

Then the argument of the max becomes:

$$
\max\!\big(\gamma_1 R - \gamma_4,\; \gamma_2 \theta_e - \gamma_3 \theta_e^* - \gamma_0\big).
$$

### Step 3: physical-space equation

Multiplying through by $\sigma$ (precipitation standard deviation) and applying the prediction framework:

$$
\boxed{P(R, \theta_e, \theta_e^*) = \max\!\Big(\exp\!\big(\sigma \cdot a \cdot \big[\max\!\big(\gamma_1 R - \gamma_4,\; \gamma_2 \theta_e - \gamma_3 \theta_e^* - \gamma_0\big)\big]^3\big) - 1,\; 0\Big).}
$$

We can further define $A \equiv \sigma \cdot a$ as the overall exponential sensitivity:

$$
P = \max\!\Big(\exp\!\Big(A \cdot \big[\max\!\big(\gamma_1 R - \gamma_4,\; \gamma_2 \theta_e - \gamma_3 \theta_e^* - \gamma_0\big)\big]^3\Big) - 1,\; 0\Big).
$$

Six physical constants with clear interpretations:

| Constant | Meaning | Formula |
|:--------:|:--------|:--------|
| $A$ | Overall exponential sensitivity | $\sigma \cdot a$ |
| $\gamma_1$ | RH sensitivity (per %) | $1 / \sigma_R$ |
| $\gamma_2$ | $\theta_e$ sensitivity (per K) | $1 / \sigma_e$ |
| $\gamma_3$ | $\theta_e^*$ sensitivity (per K) | $b / \sigma_{e^*}$ |
| $\gamma_4$ | RH offset (standardized mean) | $\mu_R / \sigma_R$ |
| $\gamma_0$ | Instability-branch offset | $\mu_e / \sigma_e - b\,\mu_{e^*} / \sigma_{e^*} + c$ |

### Physical interpretation

The max selects the **dominant driver** at each grid point and time step: either column-mean relative humidity (Branch 1), or thermodynamic instability measured by $\theta_e - \gamma_3/\gamma_2 \cdot \theta_e^*$ (Branch 2). The ratio $\gamma_3 / \gamma_2 = b\,\sigma_e / \sigma_{e^*}$ sets the relative weight of saturation equivalent potential temperature in the instability measure.

In [ ]:
name = 'sr_atm_eq'
C = REGISTRY[name]['constants']
a,b,c = C['a'],C['b'],C['c']
mu_R,sig_R   = VARMETA['rh']['mean'],VARMETA['rh']['std']
mu_e,sig_e   = VARMETA['thetae']['mean'],VARMETA['thetae']['std']
mu_es,sig_es = VARMETA['thetaestar']['mean'],VARMETA['thetaestar']['std']

A  = SIGMA * a
g1 = 1 / sig_R
g2 = 1 / sig_e
g3 = b / sig_es
g4 = mu_R / sig_R
g0 = mu_e/sig_e - b*mu_es/sig_es + c

print(f'SR-ATM physical constants (a={a}, b={b}, c={c}):')
print(f'  A      = sigma * a                  = {A:.6f}')
print(f'  gamma1 = 1 / sig_R                  = {g1:.6f} %^-1')
print(f'  gamma2 = 1 / sig_e                  = {g2:.6f} K^-1')
print(f'  gamma3 = b / sig_es                 = {g3:.6f} K^-1')
print(f'  gamma4 = mu_R / sig_R               = {g4:.6f}')
print(f'  gamma0 = mu_e/sig_e - b*mu_es/sig_es + c = {g0:.6f}')
print()
print(f'  Instability weight ratio gamma3/gamma2 = {g3/g2:.4f}')

## SR-SFC: surface fluxes as additive correction

### Normalized form

$$
f(\hat{\mathbf{x}}) = f_{\mathrm{ATM}}(\hat{R}, \hat{\theta}_e, \hat{\theta}_e^*) + a\,\widehat{\mathrm{SHF}}\,(b - \mathrm{LF}) + c\,\widehat{\mathrm{LHF}},
$$

where $f_{\mathrm{ATM}}$ is the SR-ATM equation, and $\mathrm{LF} \in [0,1]$ is the **land fraction** (not standardized).

### Step 1: substitute the SHF standardization

$$
\widehat{\mathrm{SHF}} = \frac{\mathrm{SHF} - \mu_S}{\sigma_S}, \qquad \widehat{\mathrm{LHF}} = \frac{\mathrm{LHF} - \mu_L}{\sigma_L}.
$$

The surface correction term:

$$
a\,\widehat{\mathrm{SHF}}\,(b - \mathrm{LF}) + c\,\widehat{\mathrm{LHF}} = \frac{a}{\sigma_S}\,(\mathrm{SHF} - \mu_S)(b - \mathrm{LF}) + \frac{c}{\sigma_L}\,(\mathrm{LHF} - \mu_L).
$$

### Step 2: expand and collect

$$
= \frac{a}{\sigma_S}\,\mathrm{SHF}\,(b - \mathrm{LF}) - \frac{a\,\mu_S}{\sigma_S}\,(b - \mathrm{LF}) + \frac{c}{\sigma_L}\,\mathrm{LHF} - \frac{c\,\mu_L}{\sigma_L}.
$$

Define physical constants:

$$
\delta_1 \equiv \frac{a}{\sigma_S}, \qquad \delta_2 \equiv \frac{c}{\sigma_L}, \qquad \delta_0(\mathrm{LF}) \equiv -\frac{a\,\mu_S}{\sigma_S}\,(b - \mathrm{LF}) - \frac{c\,\mu_L}{\sigma_L}.
$$

Note that $b$ here is a **fitted constant** (the critical land fraction), not the same $b$ as in SR-BL.

### Step 3: physical-space equation

$$
\boxed{P = \max\!\Big(\exp\!\Big(\sigma\big[f_{\mathrm{ATM}} + \delta_1\,\mathrm{SHF}\,(b - \mathrm{LF}) + \delta_2\,\mathrm{LHF} + \delta_0(\mathrm{LF})\big]\Big) - 1,\; 0\Big),}
$$

where $f_{\mathrm{ATM}}$ is the SR-ATM activation (in standardized space), and $\sigma$ is the precipitation standard deviation.

| Constant | Meaning | Formula |
|:--------:|:--------|:--------|
| $\delta_1$ | SHF sensitivity (W$^{-1}$ m$^2$) | $a / \sigma_S$ |
| $b$ | Critical land fraction | fitted constant |
| $\delta_2$ | LHF sensitivity (W$^{-1}$ m$^2$) | $c / \sigma_L$ |
| $\delta_0$ | LF-dependent offset | $-a\mu_S(b-\mathrm{LF})/\sigma_S - c\mu_L/\sigma_L$ |

### Physical interpretation

The surface correction has two terms: (1) sensible heat flux modulated by whether the surface is land or ocean ($b$ sets the crossover), and (2) a direct latent heat flux contribution. Over ocean ($\mathrm{LF} = 0$), the SHF term is $\delta_1 \cdot \mathrm{SHF} \cdot b > 0$; over land ($\mathrm{LF} = 1$), it becomes $\delta_1 \cdot \mathrm{SHF} \cdot (b - 1)$, which flips sign if $b < 1$.

In [ ]:
name = 'sr_sfc_eq'
C = REGISTRY[name]['constants']
a,b,c = C['a'],C['b'],C['c']
mu_S,sig_S = VARMETA['shf']['mean'],VARMETA['shf']['std']
mu_L,sig_L = VARMETA['lhf']['mean'],VARMETA['lhf']['std']

d1 = a / sig_S
d2 = c / sig_L

print(f'SR-SFC physical constants (a={a}, b={b}, c={c}):')
print(f'  delta1 = a / sig_S    = {d1:.6f} W^-1 m^2')
print(f'  b      = critical LF  = {b}')
print(f'  delta2 = c / sig_L    = {d2:.6f} W^-1 m^2')
print()
print(f'  delta0(ocean) = {-a*mu_S/sig_S * b - c*mu_L/sig_L:.6f}')
print(f'  delta0(land)  = {-a*mu_S/sig_S * (b-1) - c*mu_L/sig_L:.6f}')

## SR-ALL and SR-ALL-PC: full model with physical constraints

### Normalized forms

**SR-ALL:**

$$
f(\hat{\mathbf{x}}) = f_{\mathrm{ATM}} + (\hat{\theta}_e + a\,\widehat{\mathrm{SHF}})\,(b - \mathrm{LF})^3 + c.
$$

**SR-ALL-PC** (physically constrained — enforces $\partial P / \partial \theta_e \geq 0$):

$$
f(\hat{\mathbf{x}}) = f_{\mathrm{ATM}} + (\hat{\theta}_e + a\,\widehat{\mathrm{SHF}})\,\big[\max(b - \mathrm{LF},\, 0)\big]^3 + c.
$$

The only difference is the $\max(\cdot, 0)$ clamp, which guarantees the cubic factor is non-negative. Since $\hat{\theta}_e + a\,\widehat{\mathrm{SHF}}$ can be positive, this ensures the correction term's sign is consistent with $\partial P / \partial \theta_e \geq 0$.

### Step 1: substitute standardizations

$$
\hat{\theta}_e = \frac{\theta_e - \mu_e}{\sigma_e}, \qquad \widehat{\mathrm{SHF}} = \frac{\mathrm{SHF} - \mu_S}{\sigma_S}.
$$

The correction term (using SR-ALL-PC form, which nests SR-ALL when $\mathrm{LF} < b$):

$$
\left(\frac{\theta_e - \mu_e}{\sigma_e} + a\,\frac{\mathrm{SHF} - \mu_S}{\sigma_S}\right) \big[\max(b - \mathrm{LF},\, 0)\big]^3 + c.
$$

### Step 2: define physical-space constants

$$
\epsilon_1 \equiv \frac{1}{\sigma_e}, \qquad \epsilon_2 \equiv \frac{a}{\sigma_S},
$$

$$
\epsilon_0 \equiv -\frac{\mu_e}{\sigma_e} - \frac{a\,\mu_S}{\sigma_S}.
$$

Then the correction factor becomes:

$$
\big(\epsilon_1\,\theta_e + \epsilon_2\,\mathrm{SHF} + \epsilon_0\big)\,\big[\max(b - \mathrm{LF},\, 0)\big]^3 + c.
$$

### Step 3: physical-space equation

$$
\boxed{P = \max\!\Big(\exp\!\Big(\sigma\Big[f_{\mathrm{ATM}} + \big(\epsilon_1\,\theta_e + \epsilon_2\,\mathrm{SHF} + \epsilon_0\big)\,\big[\max(b - \mathrm{LF},\, 0)\big]^3 + c\Big]\Big) - 1,\; 0\Big).}
$$

| Constant | Meaning | Formula |
|:--------:|:--------|:--------|
| $\epsilon_1$ | $\theta_e$ sensitivity in correction (K$^{-1}$) | $1 / \sigma_e$ |
| $\epsilon_2$ | SHF sensitivity in correction (W$^{-1}$ m$^2$) | $a / \sigma_S$ |
| $\epsilon_0$ | Mean-removal offset | $-\mu_e / \sigma_e - a\mu_S / \sigma_S$ |
| $b$ | Critical land fraction | fitted constant |
| $c$ | Additive log-space offset | fitted constant |

### Physical interpretation

The correction to SR-ATM is a **land–ocean contrast** term: it is active only where $\mathrm{LF} < b$ (ocean or coastal points), modulated by a linear combination of $\theta_e$ and SHF. The cubic $(b - \mathrm{LF})^3$ means the correction strengthens sharply over open ocean. The PC form analytically guarantees the correction vanishes over land ($\mathrm{LF} \geq b$), ensuring that increased $\theta_e$ always increases precipitation — a physical constraint that the unconstrained SR-ALL can violate when $b - \mathrm{LF} < 0$.

### Key difference from SR-ALL

In SR-ALL, when $\mathrm{LF} > b$, the cubic $(b - \mathrm{LF})^3 < 0$. This means an increase in $\theta_e$ can *decrease* precipitation over land — violating PC3. The $\max(\cdot, 0)$ clamp in SR-ALL-PC eliminates this by zeroing the correction over land, deferring entirely to $f_{\mathrm{ATM}}$ in those regions.

In [ ]:
for name in ['sr_all_eq','sr_all_pc_eq']:
    if name not in REGISTRY:
        print(f'{name}: not yet in registry (constants pending NERSC optimization)\n')
        continue
    C = REGISTRY[name]['constants']
    a,b,c = C['a'],C['b'],C['c']
    mu_e,sig_e = VARMETA['thetae']['mean'],VARMETA['thetae']['std']
    mu_S,sig_S = VARMETA['shf']['mean'],VARMETA['shf']['std']

    e1 = 1 / sig_e
    e2 = a / sig_S
    e0 = -mu_e/sig_e - a*mu_S/sig_S

    label = LABELS.get(name,name)
    print(f'{label} physical constants (a={a}, b={b}, c={c}):')
    print(f'  epsilon1 = 1 / sig_e       = {e1:.6f} K^-1')
    print(f'  epsilon2 = a / sig_S        = {e2:.6f} W^-1 m^2')
    print(f'  epsilon0 = -(mu_e/sig_e + a*mu_S/sig_S) = {e0:.6f}')
    print(f'  b        = critical LF      = {b}')
    print(f'  c        = log-space offset  = {c}')
    print()

## Summary

Each SR equation converts from normalized to physical space by substituting the standardization transforms and folding training-set statistics into a small set of interpretable constants. The training-set mean $\mu$ of $\log(1+P)$ cancels entirely in the prediction pipeline — only the standard deviation $\sigma$ and the per-variable standardization parameters ($\mu_X$, $\sigma_X$) appear in the final expressions.

| Equation | Physical form | # Constants |
|:---------|:-------------|:-----------:|
| SR-BL | $P = \max\big(\exp(\alpha(B_L - B_L^*)^3 + \beta) - 1,\, 0\big)$ | 3 |
| SR-ATM | $P = \max\big(\exp(A[\max(\gamma_1 R - \gamma_4,\, \gamma_2 \theta_e - \gamma_3 \theta_e^* - \gamma_0)]^3) - 1,\, 0\big)$ | 6 |
| SR-SFC | SR-ATM + surface flux correction with $\delta_1$, $\delta_2$, $\delta_0(\mathrm{LF})$ | 6 + 4 |
| SR-ALL | SR-ATM + land–ocean contrast: $(\epsilon_1 \theta_e + \epsilon_2\,\mathrm{SHF} + \epsilon_0)(b - \mathrm{LF})^3 + c$ | 6 + 5 |
| SR-ALL-PC | Same as SR-ALL with $\max(b - \mathrm{LF},\, 0)$ enforcing PC3 | 6 + 5 |